# Multivariate and Streaming Anomaly Detection Tutorial #
This notebook is the first stage of a predictive maintenance system. It teaches how to spot unusual behaviour in a stream of sensor readings, before we know what the faults are.<br>
<br>
We use simulated pump data with fixed faults, so we know exactly where the anomalies are and can measure how well each method finds them.<br>
<br>
The notebook covers:
  - Types of anomaly: point, contextual, and collective
  - Learning normal behaviour from data with no faults in it
  - Univariate z-scores and why they miss some faults
  - Mahalanobis distance for correlated sensors
  - PCA reconstruction error
  - Streaming detectors: EWMA, CUSUM, and Page-Hinkley
  - Rolling-window features
  - Evaluation: precision, recall, point-adjusted F1, event recall, detection delay, and precision-recall curves
<br>

## Theory ##

An **anomaly** is a reading that does not fit the pattern of normal behaviour. In predictive maintenance, anomalies are the early signs of a fault. Finding them early gives time to act.

**Types of anomaly**
  - **Point anomaly:** a single reading is unusual on its own, such as a sudden spike.
  - **Contextual anomaly:** a reading is unusual for the moment it occurs. A temperature of 60 is normal under load and abnormal at idle.
  - **Collective anomaly:** each reading looks normal, but a sequence of them is not. A sensor drifting slowly upward is an example.

**Semi-supervised setting**

Labeled failures are often rare or missing. The methods here learn only from **normal** data, then flag anything that departs from it. This is called semi-supervised or one-class anomaly detection.

**Univariate z-score**

For each sensor, measure how many standard deviations a reading is from its normal mean:

$$
z_{j,t} = \frac{x_{j,t} - \mu_j}{\sigma_j}
$$

The score for a time step is the largest $|z|$ across sensors. This checks each sensor separately, so it cannot see relationships between sensors.

**Mahalanobis distance**

Sensors are usually correlated. A pump with high pressure is normally also running at high current. Mahalanobis distance measures how far a point is from the normal centre, taking that correlation into account:

$$
d^2_t = (\mathbf{x}_t - \boldsymbol{\mu})^\top \Sigma^{-1} (\mathbf{x}_t - \boldsymbol{\mu})
$$

Here $\Sigma$ is the normal covariance matrix. If the data are Gaussian, $d^2$ follows a chi-square distribution with $p$ degrees of freedom, where $p$ is the number of sensors.

Plain Euclidean distance is the straight-line gap between two points. That treats every direction the same. Sensor data isn't like that. Temperature and pressure rise together in a pump, so the normal readings form a tilted cloud, not a round one. A reading that is 3 degrees above the average may be perfectly normal if pressure is also up, but alarming if pressure is flat.

Mahalanobis distance measures how far a reading is from the normal centre, in units of normal variation. It stretches the measuring scale along the directions where the normal data spread out, and shrinks it along the directions where the data are tightly packed. Readings that follow the normal pattern get small distances. Readings that break the pattern get large ones, even when each sensor looks fine on its own.

A helpful picture: draw a ellipse around the normal cloud, sized so that most normal readings fall inside it. Mahalanobis distance tells you how far a point is outside that ellipse, measured in the ellipse's own units. A point inside the ellipse is normal. A point far outside is an anomaly.

**PCA reconstruction error**

Principal component analysis finds the directions along which normal data vary most. Keep the top $k$ directions, project each reading onto them, and map it back. Normal readings are rebuilt well. A reading that breaks the normal pattern is not:

$$
e_t = \lVert \mathbf{z}_t - V_k V_k^\top \mathbf{z}_t \rVert^2
$$

Here $\mathbf{z}_t$ is the standardized reading and $V_k$ holds the top $k$ principal directions, learned from normal data.

When choosing $k$, Too small loses normal variation, too large absorbs anomalies, aim for 90-95% of the variance explained in the principal components.  A scree plot can help show where the drop-off is.

**Statistical process control for streams**

Control charts watch one stream and react as each value arrives.
  - **EWMA** (exponentially weighted moving average) smooths the stream with weight $\lambda$ and alarms when the smoothed value leaves a band:
$$
\hat{z}_t = \lambda x_t + (1 - \lambda)\hat{z}_{t-1}
$$
  - **CUSUM** (cumulative sum) adds up deviations beyond a slack $k$ and alarms when the sum passes a limit $h$. It is good at small, sustained shifts.
  - **Page-Hinkley** tracks the cumulative deviation from the running mean and alarms when it rises too far above its lowest point. It is a standard test for drift.

**Thresholds**

A score becomes an alarm when it passes a threshold. A common choice is a high quantile of the scores on normal training data, such as the 99.9th percentile. A higher threshold means fewer false alarms and more missed faults.

**Evaluating detectors**
  - **Precision:** of the alarms raised, the fraction that are real faults
  - **Recall:** of the faulty readings, the fraction that were flagged
  - **Point-adjusted F1:** if any reading in an event is flagged, count the whole event as found. This is common in papers, but it can overstate performance, because one lucky alarm covers the whole event.
  - **Event recall** and **detection delay:** did the detector find each event, and how many samples passed before it did? For maintenance, a late alarm can be useless.
  - **Precision-recall curve:** precision and recall at every threshold. Use it when faults are rare, because accuracy hides how the detector behaves on the rare class.

**Real-world use**

Anomaly detection on sensor streams runs in many settings:
  - **Wind turbines:** a gearbox's vibration and temperature move together. A bearing fault can break that pattern before any single sensor passes its alarm limit.
  - **Data centres:** cooling fan speed, power draw, and inlet temperature are checked together, so a failing fan is flagged before servers overheat.
  - **Manufacturing:** a CNC spindle's load and vibration are monitored with control charts. Shewhart introduced these charts in the 1920s, and EWMA and CUSUM charts are later refinements.

## Simulated Pump Data ##

We simulate a pump with four sensors: temperature, pressure, vibration, and current. A slowly moving shared **load** drives all four, so the sensors are correlated. Each sensor also has its own noise.

The first 2000 readings are normal. They train the detectors. The next 1500 readings are a test stream with four labeled faults.

### Build the data ###

In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

# Fixed seed, so every run gives the same data
rng = np.random.default_rng(42)

In [ ]:
# Sensor names, normal means, and how strongly each sensor follows the shared load
SENSORS = ["temperature", "pressure", "vibration", "current"]
BASE = np.array([60.0, 4.0, 1.5, 12.0])
LOADINGS = np.array([1.0, 0.8, -0.6, 0.5])
SENSOR_NOISE = 0.3

N_TRAIN, N_TEST = 2000, 1500
N = N_TRAIN + N_TEST

# A shared load that moves slowly over time (an AR(1) process)
load = np.zeros(N)
for t in range(1, N):
    load[t] = 0.8 * load[t - 1] + 0.5 * rng.normal()

# Each sensor = baseline + its share of the load + its own noise
X = BASE + np.outer(load, LOADINGS) + rng.normal(0, SENSOR_NOISE, size=(N, 4))

# Split into normal training data and a test stream
X_train = X[:N_TRAIN].copy()
X_test = X[N_TRAIN:].copy()
print("training readings:", X_train.shape, " test readings:", X_test.shape)

### Inject the faults ###

Each fault is a labeled event in the test stream. Each one is a different shape:
  - **Spike:** a short, sharp jump in current. A point anomaly.
  - **Level shift:** temperature sits 4 degrees high for a while. A contextual anomaly, and it moves one sensor.
  - **Correlation break:** pressure is replaced by independent noise with the same mean and spread. Each sensor looks normal on its own, but the sensors no longer move together. A collective anomaly.
  - **Slow drift:** vibration ramps upward gradually. Another collective anomaly.

In [ ]:
# Start and end index of each fault, in test-stream time
EVENTS = {
    "spike": (300, 303),
    "level shift": (600, 700),
    "correlation break": (1000, 1150),
    "slow drift": (1300, 1450),
}

y_test = np.zeros(N_TEST, dtype=bool)

# Spike: current jumps for three samples
s, e = EVENTS["spike"]
X_test[s:e, 3] += 5.0

# Level shift: temperature is 4 degrees high
s, e = EVENTS["level shift"]
X_test[s:e, 0] += 4.0

# Correlation break: pressure becomes independent of the load,
# with the same mean and spread as normal data
s, e = EVENTS["correlation break"]
X_test[s:e, 1] = BASE[1] + rng.normal(0, X_train[:, 1].std(), size=e - s)

# Slow drift: vibration ramps up to 2.5 standard deviations, then resets
s, e = EVENTS["slow drift"]
X_test[s:e, 2] += np.linspace(0, 2.5 * X_train[:, 2].std(), e - s)

# Mark every faulty reading
for s, e in EVENTS.values():
    y_test[s:e] = True

print("faulty readings:", y_test.sum(), "of", N_TEST)

In [ ]:
# Plot the test stream, with faults shaded in red
fig, axes = plt.subplots(4, 1, figsize=(12, 9), sharex=True)
for i, ax in enumerate(axes):
    ax.plot(X_test[:, i], lw=0.8)
    for s, e in EVENTS.values():
        ax.axvspan(s, e, color="red", alpha=0.15)
    ax.set_ylabel(SENSORS[i])
axes[0].set_title("Test stream with labeled faults (shaded)")
axes[-1].set_xlabel("time step")
plt.tight_layout()
plt.show()

### Learn normal behaviour ###

Every detector below is fitted to the training data only. The test stream is never used to set the means, spreads, or directions.

In [ ]:
# Normal mean, spread, and covariance, from training data only
mu = X_train.mean(axis=0)
sd = X_train.std(axis=0)
cov = np.cov(X_train, rowvar=False)

print("mean:", np.round(mu, 2))
print("std: ", np.round(sd, 2))
print("correlation between sensors:")
print(np.round(np.corrcoef(X_train, rowvar=False), 2))

### Threshold from normal data ###

We set each threshold so that about 0.1% of normal training scores would be flagged. This uses one number for all detectors, so their results can be compared.

In [ ]:
# Pick the threshold as a high quantile of scores on normal training data
def normal_threshold(train_scores, q=0.999):
    return np.quantile(train_scores, q)

## Univariate Baseline ##

The simplest approach checks each sensor against its own normal range. If any sensor's z-score is too large, we raise an alarm.

**Real-world example:** many factories set a fixed alarm limit on every sensor, such as "alarm if temperature exceeds 80 degrees". This is univariate monitoring. It is easy to explain, but it cannot see when the sensors stop agreeing with each other.

In [ ]:
def zscore_scores(X, mu, sd):
    # Largest absolute z-score across the sensors, at each time step
    return np.max(np.abs((X - mu) / sd), axis=1)

z_train = zscore_scores(X_train, mu, sd)
z_test = zscore_scores(X_test, mu, sd)
thr_z = normal_threshold(z_train)
alarm_z = z_test > thr_z

print("threshold:", round(thr_z, 2))

## Mahalanobis Distance ##

Mahalanobis distance uses the full covariance matrix. A reading is far from normal if it is unusual in any direction the normal data would not produce. Two readings with the same z-scores can have very different Mahalanobis distances when one of them breaks the sensor correlation.

Here we also check the chi-square theory against the empirical threshold.

**Real-world example:** a compressor's inlet pressure and outlet pressure normally rise and fall together. If the gap between them grows, the pair is no longer behaving normally, even when each pressure is in range.

In [ ]:
def mahalanobis_sq(X, mu, cov):
    # Squared distance from the normal centre, scaled by the normal covariance
    cov_inv = np.linalg.inv(cov)
    diff = X - mu
    return np.einsum("ij,jk,ik->i", diff, cov_inv, diff)

m_train = mahalanobis_sq(X_train, mu, cov)
m_test = mahalanobis_sq(X_test, mu, cov)
thr_m = normal_threshold(m_train)
alarm_m = m_test > thr_m

# Theory says d^2 follows chi-square with 4 degrees of freedom, if the data are Gaussian
print("empirical 99.9% threshold:", round(thr_m, 2))
print("chi-square 99.9% threshold:", round(stats.chi2.ppf(0.999, df=4), 2))

The plot below shows the correlation break. Normal readings sit along a line, so pressure follows temperature. During the break, pressure is spread out and has no link to temperature. Both sensors still look normal on their own.

In [ ]:
# Compare normal readings with the correlation-break window
s, e = EVENTS["correlation break"]

fig, ax = plt.subplots(figsize=(6, 5))
ax.scatter(X_train[:, 0], X_train[:, 1], s=5, alpha=0.4, label="normal training data")
ax.scatter(X_test[s:e, 0], X_test[s:e, 1], s=8, color="red", label="correlation break")
ax.set_xlabel("temperature")
ax.set_ylabel("pressure")
ax.legend()
plt.show()

## PCA Reconstruction Error ##

PCA learns the main directions of normal variation. Our four sensors really vary along one shared load, so the first principal component should capture most of the normal behaviour. We keep one component, rebuild each reading from it, and measure what is left over.

**Real-world example:** a motor's current and speed are linked through its load. PCA learns that link from healthy data. When a reading cannot be rebuilt from the learned pattern, that residual is the alarm.

In [ ]:
def fit_pca(X, mu, sd, k):
    # Top-k principal directions of the standardized normal data
    Z = (X - mu) / sd
    _, _, Vt = np.linalg.svd(Z, full_matrices=False)
    return Vt[:k]

def pca_reconstruction_error(X, mu, sd, V):
    # Squared size of what the top-k directions cannot rebuild
    Z = (X - mu) / sd
    recon = Z @ V.T @ V
    return np.sum((Z - recon) ** 2, axis=1)

V = fit_pca(X_train, mu, sd, k=1)
print("first principal direction:", np.round(V[0], 2))

p_train = pca_reconstruction_error(X_train, mu, sd, V)
p_test = pca_reconstruction_error(X_test, mu, sd, V)
thr_p = normal_threshold(p_train)
alarm_p = p_test > thr_p

## Streaming Detectors ##

The methods above score readings in a batch. In a live system, readings arrive one at a time and each decision must be made immediately, using only the past. We write the streaming detectors as Python generators. Each one takes a stream of values and yields one score per value, using only the readings so far.

Each stream detector watches one sensor, and each is suited to a different fault shape:
  - **EWMA** on temperature, for the level shift
  - **CUSUM** on current, for a sustained shift
  - **Page-Hinkley** on vibration, for the slow drift

Inputs are standardized with the normal mean and spread, so the scores mean the same thing for every sensor.

Textbook control limits assume readings are independent. Our load signal changes slowly, so readings are strongly autocorrelated, and textbook limits fire far too often. We therefore set each threshold from normal training scores, the same rule as the other detectors.

In [ ]:
def ewma_scores(stream, lam=0.2):
    # Exponentially weighted moving average of the stream, as a size (absolute value)
    z = 0.0
    for x in stream:
        z = lam * x + (1 - lam) * z
        yield abs(z)

def cusum_scores(stream, k=0.5):
    # Upper CUSUM: builds up when readings sit above the mean by more than the slack k
    s = 0.0
    for x in stream:
        s = max(0.0, s + x - k)
        yield s

def page_hinkley_scores(stream, delta=0.1):
    # Page-Hinkley: cumulative deviation above the running mean, minus its lowest point so far
    n, mean, m, m_min = 0, 0.0, 0.0, 0.0
    for x in stream:
        n += 1
        mean += (x - mean) / n           # running mean of the stream so far
        m += x - mean - delta            # cumulative deviation from that mean
        m_min = min(m_min, m)
        yield m - m_min

In [ ]:
# Score each streaming detector on its sensor, one reading at a time.
# Each threshold comes from the normal training stream.
Z_train = (X_train - mu) / sd
Z_test = (X_test - mu) / sd

streams = [
    ("EWMA (temperature)", ewma_scores, 0),
    ("CUSUM (current)", cusum_scores, 3),
    ("Page-Hinkley (vibration)", page_hinkley_scores, 2),
]

stream_results = {}
for name, detector, col in streams:
    train_scores = np.array(list(detector(Z_train[:, col])))
    test_scores = np.array(list(detector(Z_test[:, col])))
    thr = normal_threshold(train_scores)
    stream_results[name] = (test_scores, thr)

fig, axes = plt.subplots(3, 1, figsize=(12, 8), sharex=True)
for ax, (name, (scores, thr)) in zip(axes, stream_results.items()):
    ax.plot(scores, lw=0.8)
    ax.axhline(thr, color="orange", ls="--", label="threshold from normal data")
    for s, e in EVENTS.values():
        ax.axvspan(s, e, color="red", alpha=0.15)
    ax.set_title(name)
    ax.legend(loc="upper left")
axes[-1].set_xlabel("time step")
plt.tight_layout()
plt.show()

The red bands are the true faults. A detector works when its score rises above the orange line during a red band, and stays below it elsewhere.

**Real-world example:** a hydraulic press is monitored with a CUSUM chart on pump pressure. Small shifts that accumulate over hours trigger a check, long before any single reading exceeds a fixed limit.

### Rolling-window features ###

Many detectors work better on features summarized over a window, not on single readings. A rolling standard deviation, for example, rises when a sensor starts to fluctuate. This turns a stream into a stream of features.

The generator below keeps the last `window` readings and yields their mean and standard deviation at each step.

In [ ]:
from collections import deque

def rolling_stats(stream, window=20):
    # Keep the last `window` readings, and yield (mean, std) once the window is full
    buf = deque(maxlen=window)
    for x in stream:
        buf.append(x)
        if len(buf) == window:
            yield np.mean(buf), np.std(buf)
        else:
            yield np.nan, np.nan

roll = np.array(list(rolling_stats(X_test[:, 3], window=20)))

fig, ax = plt.subplots(figsize=(12, 3.5))
ax.plot(roll[:, 1], lw=0.8)
for s, e in EVENTS.values():
    ax.axvspan(s, e, color="red", alpha=0.15)
ax.set_title("Rolling 20-step standard deviation of current")
ax.set_xlabel("time step")
plt.show()

The spike stands out in the rolling standard deviation for about 20 steps, because the window holds the spike. The window length sets how fast the feature responds and how long its effect lasts. A short window reacts quickly but is noisy. A long window is smooth but slow.

## Evaluation ##

Next we compare the three multivariate detectors on the same test stream. We report:
  - **Pointwise precision, recall, and F1:** every reading counts separately
  - **Point-adjusted F1:** an event counts as found if any of its readings is flagged
  - **False alarm rate:** the fraction of normal test readings flagged
  - **Events found and detection delay:** how many of the four faults were detected, and how many samples passed before the first alarm in each

**Real-world example:** a maintenance team cares whether a fault was caught in time, not only how many individual readings were flagged. A detector that flags the fault one hour late can be less useful than one with a slightly higher false alarm rate.

In [ ]:
def pointwise_metrics(alarms, labels):
    # Precision, recall, and F1 over individual readings
    tp = np.sum(alarms & labels)
    fp = np.sum(alarms & ~labels)
    fn = np.sum(~alarms & labels)
    precision = tp / (tp + fp) if tp + fp > 0 else 0.0
    recall = tp / (tp + fn) if tp + fn > 0 else 0.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall > 0 else 0.0
    return precision, recall, f1

def point_adjust(alarms):
    # If any reading inside an event is flagged, count the whole event as flagged
    adjusted = alarms.copy()
    for s, e in EVENTS.values():
        if alarms[s:e].any():
            adjusted[s:e] = True
    return adjusted

def event_results(alarms):
    # For each event: the number of samples before the first alarm, or None if missed
    results = {}
    for name, (s, e) in EVENTS.items():
        hits = np.flatnonzero(alarms[s:e])
        results[name] = int(hits[0]) if hits.size else None
    return results

In [ ]:
methods = {
    "z-score (per sensor)": alarm_z,
    "Mahalanobis distance": alarm_m,
    "PCA reconstruction": alarm_p,
}

print(f"{'method':<24}{'precision':>10}{'recall':>8}{'F1':>7}{'adj. F1':>9}{'false alarm':>13}")
for name, alarms in methods.items():
    p, r, f1 = pointwise_metrics(alarms, y_test)
    _, _, f1_adj = pointwise_metrics(point_adjust(alarms), y_test)
    far = alarms[~y_test].mean()
    print(f"{name:<24}{p:>10.2f}{r:>8.2f}{f1:>7.2f}{f1_adj:>9.2f}{far:>13.3%}")

print()
for name, alarms in methods.items():
    results = event_results(alarms)
    found = sum(v is not None for v in results.values())
    print(f"{name}: {found} of {len(EVENTS)} events found")
    for event, delay in results.items():
        status = "missed" if delay is None else f"first alarm after {delay} steps"
        print(f"    {event:<18} {status}")

print()
print(f"{'streaming detector':<28}{'false alarm':>13}")
for name, (scores, thr) in stream_results.items():
    alarms = scores > thr
    print(f"{name:<28}{alarms[~y_test].mean():>13.3%}")
    for event, delay in event_results(alarms).items():
        status = "missed" if delay is None else f"first alarm after {delay} steps"
        print(f"    {event:<18} {status}")

The point-adjusted F1 is always higher than the pointwise F1, and the gap can be large. Point adjustment rewards a single lucky alarm for the whole event. Event recall and detection delay give a more honest picture.

The streaming detectors have higher false alarm rates than their training thresholds suggest. Their scores are strongly autocorrelated, so 2000 training readings contain only a few independent excursions. The 99.9% quantile is therefore estimated from very little information, and new normal data exceeds it more often. In practice, streaming thresholds need a longer normal period, or periodic recalibration, before they can be trusted.

### Precision-recall curves ###

Faults are rare here, so we compare detectors across all thresholds with a precision-recall curve. Average precision is the area under that curve. A random detector scores about the fraction of faulty readings, shown as the dashed line.

In [ ]:
def precision_recall(scores, labels):
    # Precision and recall for every threshold, from high to low
    order = np.argsort(-scores)
    hits = labels[order]
    tp = np.cumsum(hits)
    fp = np.cumsum(~hits)
    precision = tp / (tp + fp)
    recall = tp / labels.sum()
    return precision, recall

def average_precision(precision, recall):
    # Area under the precision-recall curve, as a step sum
    return np.sum(np.diff(np.concatenate([[0.0], recall])) * precision)

fig, ax = plt.subplots(figsize=(6, 5))
for name, scores in [("Mahalanobis distance", m_test), ("PCA reconstruction", p_test),
                     ("z-score (per sensor)", z_test)]:
    prec, rec = precision_recall(scores, y_test)
    ap = average_precision(prec, rec)
    ax.plot(rec, prec, label=f"{name} (AP = {ap:.2f})")
ax.axhline(y_test.mean(), color="grey", ls="--", label="random")
ax.set_xlabel("recall")
ax.set_ylabel("precision")
ax.legend()
plt.show()

## Summary ##

  - **Anomalies come in shapes.** Point, contextual, and collective anomalies need different detectors. A single method rarely catches them all.
  - **Univariate checks miss relationships.** The z-score detector cannot see the correlation break, because each sensor stays in range.
  - **Mahalanobis distance and PCA reconstruction use correlation.** They catch the break, and they are the first step toward multivariate monitoring.
  - **Streaming detectors react as data arrives.** EWMA, CUSUM, and Page-Hinkley each suit a different fault shape. They use only past readings, so they can run live. Their thresholds are harder to set, because autocorrelated scores make false alarms more likely than the training data suggests.
  - **Evaluate with care.** Point-adjusted F1 can overstate results. Event recall, detection delay, and the precision-recall curve show what a maintenance team actually gets.
  - **Thresholds come from normal data.** Set the threshold from a high quantile of normal training scores, and check the false alarm rate on normal test readings.

The detectors here are tuned to this simulated data. Real sensors add missing values, sensor drift, and changing operating conditions. Expect the threshold and the streaming limits to need tuning on the real data.

## Further Reading ##
  - The `05 - Anomaly Detection` folder of this repo, which covers the univariate and threshold methods that this notebook builds on
  - SKAB (Skoltech Anomaly Benchmark): a multivariate water pump dataset with labeled anomalies, a natural next dataset for this notebook
  - Montgomery, *Introduction to Statistical Quality Control*, for EWMA and CUSUM charts
  - Ahmad et al. (2017), *Unsupervised real-time anomaly detection for streaming data*, for the Numenta detector and streaming evaluation
  - stumpy documentation: https://stumpy.readthedocs.io, for Matrix Profile discord detection, the next detector to try
  - river documentation: https://riverml.xyz, for incremental and half-space-tree detectors